# Standalone QKD Poisoning and Quantum-Noise Experiments

This Google Colab notebook is **independent of the original repository**. It contains the complete executable experiment code:

- BB84-style mismatch-count simulation with intercept-resend Eve.
- Four poisoning approaches (A–D) at five poisoning levels.
- Six machine-learning classifiers.
- The 12 original experiment figure types.
- One additional quantum-noise figure for an amplitude-damping channel.
- Automatic PNG/PDF export and a ZIP download of all figures.

The notebook uses a smaller default experiment than the paper-scale run so it can execute comfortably in Colab. Increase `N_BATCHES` after a successful test if higher statistical precision is required. Every figure is generated from values computed in this notebook; no repository files or precomputed results are read.

In [ ]:
# Install only the packages needed by this standalone notebook.
!pip -q install numpy pandas matplotlib scikit-learn

import os
import time
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, roc_curve

OUT = Path("/content/qkd_figures")
OUT.mkdir(exist_ok=True)
SEED = 42
KEY_LENGTH = 8
KEYS_PER_BATCH = 120
N_BATCHES = 500
AGGREGATION_WINDOW = 5
APPROACHES = ["A", "B", "C", "D"]
APPROACH_NAMES = {
    "A": "A: Random resample",
    "B": "B: Random +/-1",
    "C": "C: Availability (+1)",
    "D": "D: Stealth (-1)",
}
POISON_LEVELS = [10, 20, 30, 40, 50]
MODEL_NAMES = ["Random Forest", "Gradient Boosting", "Logistic Regression", "SVM", "KNN", "Neural Network"]
print("Output directory:", OUT)
print("Default experiment:", N_BATCHES, "batches,", KEYS_PER_BATCH, "keys per batch")

## 1. Complete experiment implementation

Each batch is represented losslessly by nine values (`count_0` through `count_8`), where `count_k` is the number of keys with exactly `k` mismatched bits. The feature vector is derived from this histogram.

In [ ]:
def histogram_features(counts):
    counts = np.asarray(counts, dtype=float)
    frac = counts / counts.sum(axis=1, keepdims=True)
    q = np.arange(KEY_LENGTH + 1) / KEY_LENGTH
    mean = frac @ q
    var = np.maximum(frac @ (q ** 2) - mean ** 2, 0)
    std = np.sqrt(var)
    safe = np.where(std < 1e-12, 1, std)
    centered = q[None, :] - mean[:, None]
    skew = np.where(std < 1e-12, 0, (frac * centered**3).sum(1) / safe**3)
    kurt = np.where(std < 1e-12, 0, (frac * centered**4).sum(1) / safe**4)
    p = np.array([__import__('math').comb(KEY_LENGTH, k) * .25**k * .75**(KEY_LENGTH-k)
                  for k in range(KEY_LENGTH + 1)])
    chi2 = counts.sum(1) * ((frac - p)**2 / p).sum(1)
    ks = np.abs(np.cumsum(frac, axis=1) - np.cumsum(p)).max(1)
    return np.column_stack([mean, std, skew, kurt, frac, chi2, ks])


def clean_histograms(n_batches, keys_per_batch, seed):
    rng = np.random.default_rng(seed)
    mismatches = rng.binomial(KEY_LENGTH, .25, size=(n_batches, keys_per_batch))
    return np.stack([(mismatches == k).sum(1) for k in range(KEY_LENGTH + 1)], axis=1)


def poison_histograms(clean, approach, poison_pct, seed):
    rng = np.random.default_rng([seed, ord(approach), poison_pct])
    out = clean.copy()
    total = out.sum(1)
    for row in range(len(out)):
        n = int(total[row] * poison_pct / 100)
        if n == 0:
            continue
        values = np.repeat(np.arange(KEY_LENGTH + 1), out[row])
        selected = rng.choice(len(values), size=n, replace=False)
        old = values[selected]
        if approach == "A":
            new = rng.integers(0, KEY_LENGTH + 1, size=n)
        elif approach == "B":
            new = np.clip(old + rng.choice([-1, 1], size=n), 0, KEY_LENGTH)
        elif approach == "C":
            new = np.minimum(old + 1, KEY_LENGTH)
        else:
            new = np.maximum(old - 1, 0)
        for before, after in zip(old, new):
            out[row, before] -= 1
            out[row, after] += 1
    return out


def aggregate(features, window):
    n = len(features) // window * window
    return features[:n].reshape(-1, window, features.shape[1]).mean(axis=1)


def evaluate_models(clean, poisoned, approach, poison_pct):
    x_clean = aggregate(histogram_features(clean), AGGREGATION_WINDOW)
    x_poison = aggregate(histogram_features(poisoned), AGGREGATION_WINDOW)
    x = np.vstack([x_clean, x_poison])
    y = np.r_[np.zeros(len(x_clean)), np.ones(len(x_poison))]
    split = int(len(x) * .8)
    rng = np.random.default_rng([SEED, ord(approach), poison_pct])
    order = rng.permutation(len(x))
    train, test = order[:split], order[split:]
    models = {
        "Random Forest": RandomForestClassifier(n_estimators=40, random_state=SEED, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEED, n_estimators=30),
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "SVM": make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "Neural Network": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=100, random_state=SEED)),
    }
    results = []
    for name, model in models.items():
        start = time.perf_counter()
        model.fit(x[train], y[train])
        probability = model.predict_proba(x[test])[:, 1]
        predicted = (probability >= .5).astype(int)
        fpr, tpr, _ = roc_curve(y[test], probability)
        results.append({
            "approach": approach, "poison_pct": poison_pct, "model": name,
            "accuracy": accuracy_score(y[test], predicted),
            "auc": roc_auc_score(y[test], probability),
            "seconds": time.perf_counter() - start,
            "confusion_matrix": confusion_matrix(y[test], predicted, labels=[0, 1]),
            "roc_fpr": fpr, "roc_tpr": tpr,
        })
    return results


clean = clean_histograms(N_BATCHES, KEYS_PER_BATCH, SEED)
records = []
for approach in APPROACHES:
    for level in POISON_LEVELS:
        poisoned = poison_histograms(clean, approach, level, SEED)
        records.extend(evaluate_models(clean, poisoned, approach, level))
results = pd.DataFrame([{k: v for k, v in r.items() if k not in ["confusion_matrix", "roc_fpr", "roc_tpr"]}
                        for r in records])
print("Completed", len(results), "model evaluations.")
display(results.head())

## 2. Figure helpers and the 12 original figures

In [ ]:
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
MODELS = MODEL_NAMES

def save_figure(fig, number, title):
    safe = title.lower().replace(" ", "_").replace("/", "_")
    stem = OUT / f"fig_{number:02d}_{safe}"
    fig.suptitle(title, fontsize=15, fontweight="bold")
    fig.set_layout_engine("constrained")
    fig.savefig(stem.with_suffix(".png"), dpi=180, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)

def metric_lines(metric, ylabel, number):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
    for ax, approach in zip(axes.flat, APPROACHES):
        subset = results[results.approach == approach]
        for color, model in zip(COLORS, MODELS):
            values = subset[subset.model == model].set_index("poison_pct")[metric]
            ax.plot(POISON_LEVELS, values.reindex(POISON_LEVELS), marker="o", color=color, label=model)
        ax.set_title(APPROACH_NAMES[approach]); ax.set_xlabel("Poisoning level (%)"); ax.grid(alpha=.25)
    axes[0, 0].set_ylabel(ylabel); axes[1, 0].set_ylabel(ylabel)
    axes[0, 1].legend(fontsize=8)
    save_figure(fig, number, f"{ylabel} by poisoning level")

metric_lines("accuracy", "Accuracy", 1)
metric_lines("auc", "AUC", 2)

In [ ]:
def heatmaps(metric, number):
    fig, axes = plt.subplots(1, len(POISON_LEVELS), figsize=(16, 5), sharey=True)
    norm = Normalize(vmin=max(0, results[metric].min() - .02), vmax=1)
    image = None
    for ax, level in zip(axes, POISON_LEVELS):
        matrix = results[results.poison_pct == level].pivot(index="model", columns="approach", values=metric)
        matrix = matrix.reindex(index=MODELS, columns=APPROACHES)
        image = ax.imshow(matrix, cmap="viridis", norm=norm, aspect="auto")
        ax.set_title(f"{level}%"); ax.set_xticks(range(4), APPROACHES); ax.set_xlabel("Approach")
        ax.set_yticks(range(6), MODELS)
        for r in range(6):
            for c in range(4):
                ax.text(c, r, f"{matrix.iloc[r,c]:.3f}", ha="center", va="center", fontsize=7, color="white")
    fig.colorbar(image, ax=axes, label=metric.upper())
    save_figure(fig, number, f"{metric.upper()} heatmaps")

heatmaps("accuracy", 3)
heatmaps("auc", 4)

In [ ]:
for metric, number in [("accuracy", 5), ("auc", 6)]:
    pivot = results[results.poison_pct == min(POISON_LEVELS)].pivot(index="approach", columns="model", values=metric)
    fig, ax = plt.subplots(figsize=(13, 5.5))
    pivot.reindex(index=APPROACHES, columns=MODELS).plot.bar(ax=ax, color=COLORS[:len(MODELS)])
    ax.set_xlabel("Poisoning approach"); ax.set_ylabel(metric.upper())
    ax.set_xticklabels([APPROACH_NAMES[a] for a in APPROACHES], rotation=20, ha="right")
    ax.grid(axis="y", alpha=.25); ax.legend(title="Model", ncol=3)
    save_figure(fig, number, f"{metric.upper()} comparison at lowest level")

for metric, number in [("accuracy", 7), ("auc", 8)]:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.boxplot([results[results.model == model][metric] for model in MODELS],
               tick_labels=MODELS, patch_artist=True)
    ax.set_ylabel(metric.upper()); ax.set_xlabel("Model"); ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=.25)
    save_figure(fig, number, f"{metric.upper()} distribution across experiments")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for color, model in zip(COLORS, MODELS):
    s = results[results.model == model].groupby("poison_pct").accuracy.mean()
    ax.plot(s.index, 1 - s, marker="o", color=color, label=model)
ax.set_xlabel("Poisoning level (%)"); ax.set_ylabel("Mean classification error (1 - accuracy)")
ax.grid(alpha=.25); ax.legend(ncol=3)
save_figure(fig, 9, "Mean classification error")

fig, ax = plt.subplots(figsize=(11, 5.5))
for color, model in zip(COLORS, MODELS):
    s = results[results.model == model].groupby("poison_pct").seconds.mean()
    ax.plot(s.index, s, marker="o", color=color, label=model)
ax.set_xlabel("Poisoning level (%)"); ax.set_ylabel("Mean training time (seconds)")
ax.grid(alpha=.25); ax.legend(ncol=3)
save_figure(fig, 10, "Training time by poisoning level")

In [ ]:
hardest = [r for r in records if r["approach"] == "B" and r["poison_pct"] == 10]
fig, ax = plt.subplots(figsize=(8, 6))
for color, row in zip(COLORS, hardest):
    ax.plot(row["roc_fpr"], row["roc_tpr"], color=color, label=f'{row["model"]} (AUC={row["auc"]:.3f})')
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
ax.grid(alpha=.25); ax.legend(fontsize=8)
save_figure(fig, 11, "ROC curves at approach B and 10%")

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, row in zip(axes.flat, hardest):
    matrix = row["confusion_matrix"]
    ax.imshow(matrix, cmap="Blues"); ax.set_title(row["model"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks([0, 1], ["Clean", "Poisoned"]); ax.set_yticks([0, 1], ["Clean", "Poisoned"])
    for r in range(2):
        for c in range(2):
            ax.text(c, r, int(matrix[r, c]), ha="center", va="center")
save_figure(fig, 12, "Confusion matrices at approach B and 10%")

## 3. Additional quantum-noise figure (Figure 13)

This figure simulates BB84 sifted-key error rates under an amplitude-damping channel without Eve. It compares analytic Z-basis, X-basis, and total QBER curves.

In [ ]:
def amplitude_damping_qber(gamma):
    q_z = gamma / 2
    q_x = (1 - np.sqrt(1 - gamma)) / 2
    return q_z, q_x, (q_z + q_x) / 2

gamma = np.linspace(0, 1, 101)
qz, qx, qt = np.array([amplitude_damping_qber(g) for g in gamma]).T
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(gamma, qz, label="Z-basis QBER", linewidth=2)
ax.plot(gamma, qx, label="X-basis QBER", linewidth=2)
ax.plot(gamma, qt, label="Total QBER", linewidth=2)
ax.set_xlabel("Amplitude-damping parameter gamma"); ax.set_ylabel("QBER")
ax.grid(alpha=.25); ax.legend()
save_figure(fig, 13, "Quantum noise amplitude damping QBER")

## 4. Download all figures

The next cell creates a ZIP archive containing all PNG and PDF figures. In Colab, it triggers a browser download.

In [ ]:
archive = Path("/content/qkd_all_figures.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.glob("*")):
        if file.suffix in {".png", ".pdf"}:
            z.write(file, arcname=file.name)
print("Created:", archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Outside Colab: download the archive from", archive)

## Figure index

1. Accuracy by poisoning level  
2. AUC by poisoning level  
3. Accuracy heatmaps  
4. AUC heatmaps  
5. Accuracy comparison at the lowest level  
6. AUC comparison at the lowest level  
7. Accuracy distribution across experiments  
8. AUC distribution across experiments  
9. Mean classification error  
10. Training time by poisoning level  
11. ROC curves at Approach B and 10% poisoning  
12. Confusion matrices at Approach B and 10% poisoning  
13. Quantum-noise amplitude-damping QBER